### Installation des dépendances

Configuration adaptée du notebook `LLM setup` fourni dans le modèle de projet pour exécuter [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) et le pipeline RAG sur Google Colab.

- L'environnement Colab est configuré avec un GPU T4 et un runtime Python 3.11.
- La cellule installe les dépendances nécessaires avec `uv`, en fixant les versions de certaines bibliothèques et en ajoutant celles nécessaires au RAG.
- L'installation est ignorée lorsque PandasAI 3.0.0 et les trois bibliothèques RAG sont déjà présents. Après une première installation, un redémarrage de la session Colab est nécessaire.
- Hors Colab, l'installation est ignorée et l'environnement peut être configuré localement avec `uv sync --extra llm --extra rag`.

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and (installed_version("pandasai") != "3.0.0" or
                 installed_version("sentence-transformers") is None or
                 installed_version("faiss-cpu") is None or
                 installed_version("langchain-text-splitters") is None):
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])

    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "requests",
        "pandasai==3.0.0",

        # RAG dependencies
        "sentence-transformers>=6.1.0",
        "faiss-cpu>=1.15.1",
        "langchain-text-splitters>=1.1.3",
    ]

    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

### Récupération du dépôt GitHub

Cette cellule clone le dépôt GitHub de notre équipe dans `/content/repo` afin de rendre le package `ift6758` accessible depuis Colab.

- Le clonage est effectué uniquement si le dossier cible n'existe pas déjà.
- Le dossier du dépôt est ajouté à `sys.path` pour permettre l'importation des modules Python du projet.
- Hors Colab, cette étape est ignorée.

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata

    REPO = "ammou35/LNH-data-science"
    REPO_DIR = "/content/repo"
    BRANCH = "llm-code-generation"

    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

### Vérification de l'environnement

- Vérification de la version de Python et des principales dépendances.
- Affichage des versions utilisées pour documenter l'environnement et faciliter la reproductibilité des résultats.

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
from importlib.metadata import version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("A Python 3.11 runtime is required.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("Accelerate:", version("accelerate"))
print("PandasAI:", version("pandasai"))

Python: 3.11.13
NumPy: 1.26.4 | pandas: 1.5.3
Transformers: 4.57.6
Sentence Transformers: 3.4.1
CUDA GPU: Tesla T4


### Vérification du GPU

- Si un GPU est disponible, son nom est affiché.
- Sinon, un message rappelle de sélectionner un runtime GPU T4 dans les paramètres de Google Colab.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).


### Chargement du modèle Qwen

- Le modèle Qwen3-4B-Instruct-2507 est chargé à l'aide de la fonction `load_model()` du package `ift6758`.
- Lors de la première exécution, environ 8 Go de poids sont téléchargés dans le cache Hugging Face.
- Le périphérique d'exécution et le type de données numériques du modèle sont affichés après le chargement.
- Une première requête sur le hockey permet de vérifier la génération de texte avec le gabarit de conversation de Qwen.

In [ ]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

In [ ]:
def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
  messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
  inputs = tokenizer.apply_chat_template(
      messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
  ).to(model.device)
  output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

  return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask_qwen("In one sentence, what is a hat trick in hockey?"))

### Préparation de la documentation NHL pour le RAG

Cette cellule prépare la documentation de l'API NHL pour rechercher les informations nécessaires à la génération du code d'acquisition.

In [ ]:
from ift6758.llm.rag import download_documentation, retrieve_sections

chunks, embedder, index = download_documentation()

### Récupération des extraits pertinents

Cette cellule utilise le pipeline RAG pour sélectionner les passages de la documentation utiles à l'acquisition des données play-by-play.

In [ ]:
questions = [
    "Which endpoint retrieves a team's game schedule for a specific NHL season?",
    "Which endpoint retrieves play-by-play data for a specific game ID?",
]

contexts = [
    retrieve_sections(question, chunks, embedder, index)
    for question in questions
]

context = "\n\n".join(contexts)
print(context)
assert "/v1/gamecenter/{game-id}/play-by-play" in context, "Inspect retrieval before generating."

### Génération du code d'acquisition avec Qwen

Cette cellule demande à Qwen3-4B-Instruct-2507 de générer une implémentation Python pour l'acquisition des données play-by-play d'une saison NHL.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

prompt = f"""NHL API documentation:
{context}

Write a Python function fetch_season_llm(season_id, cache_dir) that retrieves,
caches, and loads play-by-play data for a given NHL season, including the
regular season and playoffs.
Requirements from our project:
- Use requests, pathlib and json. Include imports and a docstring.
- The API host is https://api-web.nhle.com.
- If a season's cached JSON file exists, load it with no network request.
- Otherwise GET its play-by-play with timeout=30 and raise_for_status().
- Each play-by-play response should have id (int) and plays (list). Validate both.
- Create the cache directory and save valid JSON as UTF-8.
- Return the play-by-play data for the season. Validate cached data too.
Return only the function and imports, without Markdown formatting.
"""

generated_code = ask_qwen(prompt, system="You write concise, correct Python code.", max_new_tokens=800)

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

generated_code_path = PACKAGE_DIR / "llm" / "question_1_llm.py"
generated_code_path.parent.mkdir(parents=True, exist_ok=True)
generated_code_path.write_text(generated_code, encoding="utf-8")

display(Markdown(generated_code))

### Vérification du code généré

Cette cellule prépare le code proposé par Qwen pour son évaluation.

In [ ]:
from ift6758.data.nhl import download_season

# Remplacer la ligne suivante par le code généré par Qwen dans la sortie de la
# cellule précédente :
fetch_season_llm = None

### Préparation du match de référence

Cette cellule récupère les données réelles d'un match NHL connu afin de disposer d'une référence indépendante pour la comparaison.

- Le match `2019020001`, appartenant à la saison `2019-2020`, est sélectionné comme référence.
- Si son fichier JSON existe déjà, les données sont chargées localement. Sinon, elles sont téléchargées depuis l'API NHL et sauvegardées.
- L'identifiant du match et la présence d'une liste d'événements sont vérifiés.
- Le dictionnaire game contient les données de référence utilisées dans les tests suivants.

In [ ]:
import json
import requests

SEASON_ID = 2019
GAME_ID = 2019020001

raw_path = DATA_DIR / "raw" / f"{GAME_ID}.json"

if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play",
        timeout=30,
    )
    response.raise_for_status()

    downloaded = response.json()
    assert downloaded["id"] == GAME_ID
    assert isinstance(downloaded["plays"], list)

    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")

game = json.loads(raw_path.read_text(encoding="utf-8"))

assert game["id"] == GAME_ID
assert isinstance(game["plays"], list)

print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

### Préparation des résultats de comparaison

Cette cellule initialise les structures utilisées pour conserver les résultats des tests.

In [ ]:
from tempfile import TemporaryDirectory
from unittest.mock import Mock, patch
from pathlib import Path
import requests

results = {
    "Manual": {
        "match_connu": False,
        "cache": False,
        "erreur_http": False,
    },
    "Qwen": {
        "match_connu": False,
        "cache": False,
        "erreur_http": False,
    },
}

### Comparaison des implémentations

Cette section compare l'implémentation manuelle de l'équipe et celle générée par Qwen à l'aide des mêmes tests :
1. un match connu
2. un accès au cache sans nouvelle requête
3. une requête qui échoue.

### Tests 1 et 2 (match connu et cache sans nouvelle requête)

In [ ]:
with TemporaryDirectory() as temp_dir:
    manual_folder = Path(temp_dir) / "manual"
    llm_folder = Path(temp_dir) / "llm"

    # 1. Match connu

      # Implémentation manuelle
    response = Mock()
    response.status_code = 200
    response.json.return_value = game

    not_found = Mock()
    not_found.status_code = 404

    session = Mock()
    session.get.side_effect = lambda url, timeout: (
        response if url.endswith(f"/{GAME_ID}/play-by-play") else not_found
    )

    try:
        manual_first = download_season(
            SEASON_ID, manual_folder, session=session, delay=0
        )

        assert any(
            call.args[0] == f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play"
            and call.kwargs.get("timeout") == 30
            for call in session.get.call_args_list
        )
        response.raise_for_status.assert_called_once()

        assert manual_first["regular"] == [game]
        assert manual_first["playoffs"] == []

        manual_saved = json.loads(
            (manual_folder / str(SEASON_ID) / f"{GAME_ID}.json").read_text(encoding="utf-8")
        )
        assert manual_saved == game

        results["Manual"]["match_connu"] = True

    except Exception:
        results["Manual"]["match_connu"] = False

    # Implémentation Qwen
    schedule_response = Mock()
    schedule_response.json.return_value = {
        "games": [{"id": GAME_ID}]
    }

    response = Mock()
    response.json.return_value = game

    try:
        with patch("requests.get", side_effect=[schedule_response, response]) as get:
            llm_first = fetch_season_llm(SEASON_ID, llm_folder)

        assert get.call_count == 2
        assert any(
            call.args[0] == f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play"
            and call.kwargs.get("timeout") == 30
            for call in get.call_args_list
        )
        schedule_response.raise_for_status.assert_called_once()
        response.raise_for_status.assert_called_once()

        assert llm_first["season"] == SEASON_ID
        assert llm_first["games"] == [game]

        llm_saved = json.loads(
            (llm_folder / f"{SEASON_ID}.json").read_text(encoding="utf-8")
        )
        assert llm_first == llm_saved

        results["Qwen"]["match_connu"] = True

    except Exception as error:
        results["Qwen"]["match_connu"] = False

    # 2. Accès au cache sans nouvelle requête

    # Implémentation manuelle
    if results["Manual"]["match_connu"]:
        try:
            session.get.reset_mock()
            session.get.side_effect = AssertionError("Cache was ignored")

            manual_second = download_season(
                SEASON_ID, manual_folder, session=session, delay=0
            )

            assert session.get.call_count == 0
            assert manual_first == manual_second
            results["Manual"]["cache"] = True

        except Exception:
            results["Manual"]["cache"] = False

    # Implémentation Qwen
    if results["Qwen"]["match_connu"]:
        try:
            with patch("requests.get", side_effect=AssertionError("Cache was ignored")):
                llm_second = fetch_season_llm(SEASON_ID, llm_folder)

            assert llm_first == llm_second == llm_saved
            results["Qwen"]["cache"] = True

        except Exception as error:
            results["Qwen"]["cache"] = False

### Test 3 (requête qui échoue)

In [ ]:
# 3. Requête HTTP qui échoue

with TemporaryDirectory() as temp_dir:
    manual_folder = Path(temp_dir) / "manual"
    llm_folder = Path(temp_dir) / "llm"

    # Implémentation manuelle
    response = Mock()
    response.status_code = 500
    response.raise_for_status.side_effect = requests.HTTPError("HTTP 500")

    session = Mock()
    session.get.return_value = response

    try:
        try:
            download_season(SEASON_ID, manual_folder, session=session, delay=0)
        except requests.HTTPError:
            session.get.assert_called_once_with(
                f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play",
                timeout=30,
            )
            response.raise_for_status.assert_called_once()
        else:
            raise AssertionError("HTTP error was not propagated.")

        assert not (manual_folder / str(SEASON_ID) / f"{GAME_ID}.json").exists()
        results["Manual"]["erreur_http"] = True

    except Exception:
        results["Manual"]["erreur_http"] = False

    # Généré par Qwen
    response = Mock()
    response.status_code = 500
    response.raise_for_status.side_effect = requests.HTTPError("HTTP 500")

    try:
        with patch("requests.get", return_value=response) as get:
            try:
                fetch_season_llm(SEASON_ID, llm_folder)
            except requests.HTTPError:
                get.assert_called()
                response.raise_for_status.assert_called()
            else:
                raise AssertionError("HTTP error was not propagated.")

        assert not (llm_folder / f"{SEASON_ID}.json").exists()
        results["Qwen"]["erreur_http"] = True

    except Exception as error:
        results["Qwen"]["erreur_http"] = False